# 🧠 Brain Tumor Classification — ResNet-50
**Your Dataset · 7,023 MRI images · 4 Classes**

| Split | Glioma | Healthy | Meningioma | Pituitary | Total |
|-------|--------|---------|------------|-----------|-------|
| Train | 1,297  | 1,600   | 1,316      | 1,405     | 5,618 |
| Val   | 162    | 200     | 164        | 176       | 702   |
| Test  | 162    | 200     | 165        | 176       | 703   |
| **Total** | **1,621** | **2,000** | **1,645** | **1,757** | **7,023** |

---
### Why ResNet-50?
| Property | Value |
|---|---|
| Parameters | ~25.6M |
| GFLOPs | ~4.1 |
| ImageNet Top-1 Acc | 76.1% (V1) |
| Input Resolution | 224×224 |
| Strategy | Deep Residual Learning (Identity Mappings) |

ResNet-50 is a industry standard for medical imaging due to its residual connections which prevent vanishing gradients in deep networks, making it highly effective for identifying subtle tumor boundaries.

## 📦 Step 1 — Install Dependencies

In [5]:
!pip install -q torchinfo thop seaborn scikit-learn matplotlib torchvision

## 📁 Step 2 — Upload & Extract Your Dataset

Upload your **Organized.rar** file (the same one you have) and extract it.  
The notebook expects the structure:
```
Organized/
  train/ glioma/ healthy/ meningioma/ pituitary/
  val/   glioma/ healthy/ meningioma/ pituitary/
  test/  glioma/ healthy/ meningioma/ pituitary/
```

In [ ]:
# ── Option A: Upload from local machine ─────────────────────────────────────
from google.colab import files
import os, subprocess

print("Upload your Organized.rar file:")
uploaded = files.upload()   # click to browse & upload

rar_name = list(uploaded.keys())[0]
print(f"\nUploaded: {rar_name}")

# Install unrar and extract
subprocess.run(['apt-get', 'install', '-y', 'unrar'], capture_output=True)
os.makedirs('/content/data', exist_ok=True)
result = subprocess.run(['unrar', 'x', '-y', rar_name, '/content/data/'], capture_output=True)
print(result.stdout.decode()[-500:] if result.stdout else "Extraction done")

DATASET_ROOT = '/content/data/Organized'
print("\nFolder structure:")
for split in ['train', 'val', 'test']:
    for cls in ['glioma', 'healthy', 'meningioma', 'pituitary']:
        path = f'{DATASET_ROOT}/{split}/{cls}'
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f"  {split}/{cls}: {count} images")

Upload your Organized.rar file:


In [ ]:
# ── Option B: Load from Google Drive (if dataset already there) ─────────────
# Uncomment and run INSTEAD of Option A if you prefer Drive

# from google.colab import drive
# drive.mount('/content/drive')
# import subprocess, os

# rar_path  = '/content/drive/MyDrive/Organized.rar'  # ← adjust path
# os.makedirs('/content/data', exist_ok=True)
# subprocess.run(['apt-get', 'install', '-y', 'unrar'], capture_output=True)
# subprocess.run(['unrar', 'x', '-y', rar_path, '/content/data/'])
# DATASET_ROOT = '/content/data/Organized'


## ☁️ Step 3 — Mount Google Drive (save checkpoints & results)

In [ ]:
from google.colab import drive
import os

# Try to get RUN_NAME from the environment, otherwise use a default
RUN_NAME ="experiment 1 aug_on LR_0.001 batchsize_8 epochs_15"

drive.mount('/content/drive')

# ── Dynamic Folder Creation based on RUN_NAME ──
DRIVE_ROOT     = f'/content/drive/MyDrive/BrainTumor/{RUN_NAME}'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR    = f'{DRIVE_ROOT}/results'

for d in [CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"All artifacts for '{RUN_NAME}' will be saved to:")
print(f"  Checkpoints : {CHECKPOINT_DIR}")
print(f"  Results     : {RESULTS_DIR}")

In [ ]:
import os

# Manual override based on your actual folder structure seen in the extraction log
DATASET_ROOT = '/content/data/Clache'
TRAIN_DIR = os.path.join(DATASET_ROOT, 'train')
VAL_DIR = os.path.join(DATASET_ROOT, 'val')
TEST_DIR = os.path.join(DATASET_ROOT, 'test')

# Verification
print(f"🔍 Checking directories in: {DATASET_ROOT}")
for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    exists = os.path.exists(folder)
    count = len(os.listdir(folder)) if exists else 0
    print(f"  {'✅' if exists else '❌'} {os.path.basename(folder)}: {count} sub-folders found")

if os.path.exists(TRAIN_DIR):
    print("\n🚀 Paths successfully locked to /content/data/Clache/")
else:
    print("\n❌ Error: Directories still not found. Please ensure the extraction in Step 2 finished correctly.")

## ⚙️ Step 4 — Hyperparameters
**Edit these values before training.**

In [ ]:
# ╔═══════════════════════════════════════════════════════╗
# ║          HYPERPARAMETER CONFIGURATION                 ║
# ╚═══════════════════════════════════════════════════════╝


EPOCHS         = 15          # total training epochs
BATCH_SIZE     = 8          # images per batch
OPTIMIZER      = "adam"      # optimizer type: 'adam' | 'sgd'
LEARNING_RATE  = 1e-3        # initial learning rate
WEIGHT_DECAY   = 1e-4        # L2 regularisation
IMAGE_SIZE     = 224         # ResNet standard resolution
NUM_CLASSES    = 4
CLASS_NAMES    = ['glioma', 'healthy', 'meningioma', 'pituitary']
NUM_WORKERS    = 4

# ── Data Augmentation Toggle ─────────────────────────────
AUGMENTATION_ON = True       # Set to False to disable random augmentations

# ── Early Stopping ───────────────────────────────────────
EARLY_STOPPING_PATIENCE  = 100
EARLY_STOPPING_MIN_DELTA = 1e-4

# ── LR Scheduler ─────────────────────────────────────────
LR_SCHEDULER = 'cosine'
LR_STEP_SIZE = 10
LR_GAMMA     = 0.1

# ── Fine-tuning strategy ──────────────────────────────────
UNFREEZE_AFTER_EPOCH = 5

SEED = 42
print(f"Hyperparameters loaded for run: {RUN_NAME} ✅")
print(f"Augmentation Status: {'ENABLED' if AUGMENTATION_ON else 'DISABLED'}")

## 🔧 Step 5 — Imports & Reproducibility

In [ ]:
import random, time, json, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchinfo import summary
from thop import profile, clever_format

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score
)

warnings.filterwarnings('ignore')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 🖼️ Step 6 — Data Augmentation & Loaders

In [ ]:
import os

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# --- Automatic Path Correction ---
# If the default path doesn't exist, we look for where the folders actually are
if not os.path.exists(f'{DATASET_ROOT}/train'):
    print(f"⚠️  Warning: '{DATASET_ROOT}/train' not found. Searching for correct path...")
    for root, dirs, files in os.walk('/content/data'):
        if 'train' in dirs and 'val' in dirs:
            DATASET_ROOT = root
            print(f"✅ Found dataset at: {DATASET_ROOT}")
            break

# Define standard transforms used for validation/testing
base_transforms = [
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
]

# Define training transforms based on the AUGMENTATION_ON toggle
if AUGMENTATION_ON:
    train_transforms_list = [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.3),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
else:
    train_transforms_list = base_transforms

train_transforms = transforms.Compose(train_transforms_list)
val_test_transforms = transforms.Compose(base_transforms)

# Re-initialize datasets with the selected transforms
TRAIN_DIR = f'{DATASET_ROOT}/train'
VAL_DIR   = f'{DATASET_ROOT}/val'
TEST_DIR  = f'{DATASET_ROOT}/test'

try:
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
    val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_test_transforms)
    test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_test_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    print(f"Class mapping : {train_dataset.class_to_idx}")
    print(f"Augmentations active in Training: {AUGMENTATION_ON}")
except FileNotFoundError as e:
    print(f"❌ Error: Could not find dataset folders in {DATASET_ROOT}.")
    print("Please ensure you uploaded and extracted the dataset correctly in Step 2.")

## 🔬 Step 9 — Model Architecture & Statistics

In [ ]:
def build_model(num_classes=4, pretrained=True):
    # 1. Update the weights enum and model builder for ResNet50
    weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet50(weights=weights)

    # 2. ResNet uses '.fc' instead of '.classifier'
    in_features = model.fc.in_features

    # Replace the final fully connected layer with the custom sequential head
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(512, num_classes)
    )
    return model

model = build_model(NUM_CLASSES).to(DEVICE)

# ── torchinfo summary ────────────────────────────────────────────────────────
print('='*70)
print('  ResNet-50  —  fine-tuned for Brain Tumor Classification')
print('='*70)
model_stats = summary(
    model,
    input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE),
    col_names=['input_size','output_size','num_params','trainable'],
    col_width=20, row_settings=['var_names'], verbose=1
)

# ── GFLOPs ───────────────────────────────────────────────────────────────────
# FIX: Ensure dummy input is on the same device as the model
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
flops, params_thop = profile(model, inputs=(dummy,), verbose=False)
flops_str, params_str = clever_format([flops, params_thop], '%.3f')
print(f'\n  GFLOPs     : {float(flops)/1e9:.3f} G')
print(f'  Parameters : {params_str}')

# ── Layer-type distribution ──────────────────────────────────────────────────
import pandas as pd
from collections import Counter

layer_counts = Counter(type(m).__name__ for m in model.modules()
                        if len(list(m.children())) == 0)
df_layers = pd.DataFrame(layer_counts.items(),
                          columns=['Layer Type','Count']).sort_values('Count', ascending=False)
print('\nLayer-type distribution:')
print(df_layers.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
cmap = plt.cm.viridis(np.linspace(0.2, 0.9, len(df_layers)))
bars = ax.barh(df_layers['Layer Type'], df_layers['Count'], color=cmap)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_xlabel('Count')
ax.set_title('ResNet-50 — Layer Type Distribution', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(f'{RESULTS_DIR}/layer_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Layer distribution saved ✅")

## 🚀 Step 10 — Training Setup (Optimizer, Loss, Scheduler)

In [ ]:
model = model.to(DEVICE)

# Weighted loss for slight class imbalance
class_counts = np.array([Counter(train_dataset.targets)[i] for i in range(NUM_CLASSES)], dtype=float)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
criterion = nn.CrossEntropyLoss(weight=class_weights)

def freeze_backbone(m):
    for name, param in m.named_parameters():
        # FIX: ResNet uses '.fc', EfficientNet uses '.classifier'
        param.requires_grad = ('fc' in name)

def unfreeze_all(m):
    for param in m.parameters():
        param.requires_grad = True

# Start by only training the new head
freeze_backbone(model)

# Configure Optimizer based on hyperparams
trainable_params = filter(lambda p: p.requires_grad, model.parameters())
if OPTIMIZER.lower() == 'adam':
    optimizer = optim.Adam(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
else:
    optimizer = optim.SGD(trainable_params, lr=LEARNING_RATE, momentum=0.9, weight_decay=WEIGHT_DECAY)

if LR_SCHEDULER == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
elif LR_SCHEDULER == 'step':
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
elif LR_SCHEDULER == 'plateau':
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  (backbone frozen for first {UNFREEZE_AFTER_EPOCH} epochs)")
print(f"Frozen params    : {total - trainable:,}")
print(f"Loss weights     : {dict(zip(CLASS_NAMES, class_weights.cpu().numpy().round(3)))}")
print(f"Optimizer        : {type(optimizer).__name__}")
print("Setup ready ✅")

## 🏋️ Step 11 — Training Loop with Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4, path='best.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.path = path
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            torch.save(model.state_dict(), self.path)
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            print(f'   ⌛ EarlyStopping: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            torch.save(model.state_dict(), self.path)
            self.counter = 0
            print(f'   ✅ Val loss improved → checkpoint saved')

def run_epoch(loader, model, criterion, optimizer=None, phase='train'):
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            if is_train:
                optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total

# Updated path to use the new CHECKPOINT_DIR
BEST_PATH = f'{CHECKPOINT_DIR}/best_resnet50_braintumor.pth'
early_stopper = EarlyStopping(
    patience=EARLY_STOPPING_PATIENCE,
    min_delta=EARLY_STOPPING_MIN_DELTA,
    path=BEST_PATH
)

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'lr':[]}
print(f"Training ResNet-50 for run '{RUN_NAME}' on {DEVICE}\n")

for epoch in range(1, EPOCHS + 1):
    if epoch == UNFREEZE_AFTER_EPOCH + 1:
        unfreeze_all(model)
        for g in optimizer.param_groups:
            g['lr'] = LEARNING_RATE * 0.1
        early_stopper.counter = 0
        early_stopper.best_score = None
        print(f"  → Backbone unfrozen, patience reset.")

    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer, 'train')
    val_loss,   val_acc   = run_epoch(val_loader,   model, criterion, None,      'val')

    scheduler.step() if LR_SCHEDULER != 'plateau' else scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    print(f"Epoch {epoch}: Train Loss {train_loss:.4f} | Val Acc {val_acc:.4f}")
    early_stopper(val_loss, model)
    if early_stopper.early_stop: break

with open(f'{RESULTS_DIR}/history.json', 'w') as f:
    json.dump(history, f)
print("\nTraining complete ✅ Results directed to Google Drive.")

## 📈 Step 12 — Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Curves — ResNet-50 · Brain Tumor Dataset',
              fontsize=13, fontweight='bold')

# Loss
axes[0].plot(epochs_ran, history['train_loss'], 'b-o', ms=3, lw=1.5, label='Train')
axes[0].plot(epochs_ran, history['val_loss'],   'r-o', ms=3, lw=1.5, label='Val')
axes[0].fill_between(epochs_ran, history['train_loss'], history['val_loss'], alpha=0.08, color='purple')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_ran, history['train_acc'], 'b-o', ms=3, lw=1.5, label='Train')
axes[1].plot(epochs_ran, history['val_acc'],   'r-o', ms=3, lw=1.5, label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend(); axes[1].grid(alpha=0.3)

# Learning Rate
axes[2].plot(epochs_ran, history['lr'], 'g-o', ms=3, lw=1.5)
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training curves saved ✅")

## 🧪 Step 13 — Evaluate on Test Set (703 images)

In [ ]:
import json

# Ensure the model used for testing is the best one from the current run's checkpoint directory
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

test_acc = accuracy_score(all_labels, all_preds)
report_text = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4)
report_dict = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)

# Save the JSON report to the dynamic RESULTS_DIR
json_report = {
    "test_accuracy": float(test_acc),
    "class_metrics": {cls: report_dict[cls] for cls in CLASS_NAMES},
    "macro_avg": report_dict["macro avg"],
    "weighted_avg": report_dict["weighted avg"]
}

with open(f'{RESULTS_DIR}/classification_report.json', 'w') as f:
    json.dump(json_report, f, indent=4)

with open(f'{RESULTS_DIR}/classification_report.txt', 'w') as f:
    f.write(f"Test Accuracy: {test_acc:.4f}\n\n{report_text}")

print(f"Reports saved to: {RESULTS_DIR} ✅")

## 📉 Step 14 — Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Confusion Matrix — Test Set  (703 images)', fontsize=13, fontweight='bold')

for ax, data, fmt, title in zip(
        axes,
        [cm, cm_norm],
        ['d', '.3f'],
        ['Raw Counts', 'Row-Normalised']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
                annot_kws={'size': 11})
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved ✅")

## 📊 Step 15 — Per-Class Metrics (Precision · Recall · F1 · Accuracy)

In [ ]:
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, labels=list(range(NUM_CLASSES)))
per_class_acc = cm.diagonal() / cm.sum(axis=1)

metrics_df = pd.DataFrame({
    'Class'     : CLASS_NAMES,
    'Precision' : precision,
    'Recall'    : recall,
    'F1-Score'  : f1,
    'Accuracy'  : per_class_acc,
    'Support'   : support.astype(int)
})

print("Per-Class Metrics on Test Set:")
print(metrics_df.to_string(index=False, float_format='{:.4f}'.format))

# ── Grouped bar chart ─────────────────────────────────────────────────────────
x = np.arange(NUM_CLASSES)
width = 0.2
metric_cols  = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
bar_colors   = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(13, 6))
for i, (col, color) in enumerate(zip(metric_cols, bar_colors)):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, metrics_df[col], width, label=col, color=color, alpha=0.87)
    ax.bar_label(bars, fmt='%.3f', fontsize=7.5, padding=2, rotation=45)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Per-Class Metrics — Test Set', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Per-class metrics chart saved ✅")

## 🕸️ Step 16 — Per-Class Radar Chart

In [ ]:
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

categories = ['Precision', 'Recall', 'F1-Score', 'Accuracy']
N = len(categories)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors_radar = ['#E74C3C', '#2ECC71', '#3498DB', '#9B59B6']

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_facecolor('#f8f9fa')

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    values = [precision[cls_idx], recall[cls_idx], f1[cls_idx], per_class_acc[cls_idx]]
    values += values[:1]
    ax.plot(angles, values, 'o-', lw=2, color=colors_radar[cls_idx], label=cls_name)
    ax.fill(angles, values, alpha=0.1, color=colors_radar[cls_idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=8)
ax.set_title('Per-Class Performance Radar — Test Set',
              fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/radar_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Radar chart saved ✅")

## 🏆 Step 17 — Overall Performance Dashboard

In [ ]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support
import os

# Ensure inputs are numpy arrays for metric calculation
y_true = np.array(all_labels)
y_pred = np.array(all_preds)
current_acc = accuracy_score(y_true, y_pred)

macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
weighted_p, weighted_r, weighted_f, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

summary_metrics = {
    'Overall Accuracy'   : current_acc,
    'Macro Precision'    : macro_p,
    'Macro Recall'       : macro_r,
    'Macro F1'           : macro_f,
    'Weighted Precision' : weighted_p,
    'Weighted Recall'    : weighted_r,
    'Weighted F1'        : weighted_f,
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Overall Model Performance — Test Set', fontsize=13, fontweight='bold')

# ── Horizontal bar chart ──────────────────────────────────────────────────────
names  = list(summary_metrics.keys())
values = list(summary_metrics.values())
bar_colors_ov = ['#2ecc71' if v >= 0.90 else '#f39c12' if v >= 0.75 else '#e74c3c' for v in values]
bars = ax1.barh(names, values, color=bar_colors_ov, alpha=0.85)
ax1.bar_label(bars, fmt='%.4f', padding=4, fontsize=11, fontweight='bold')
ax1.set_xlim(0, 1.12)
ax1.set_xlabel('Score', fontsize=11)
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)
ax1.set_title('Aggregate Metrics', fontsize=11)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='≥ 0.90 (Excellent)'),
    Patch(facecolor='#f39c12', label='≥ 0.75 (Good)'),
    Patch(facecolor='#e74c3c', label='< 0.75 (Needs Work)'),
]
ax1.legend(handles=legend_elements, loc='lower right', fontsize=9)

# ── Support pie chart ──────────────────────────────────────────────────────────
label_counts = Counter(y_true)
test_counts = [label_counts[i] for i in range(len(CLASS_NAMES))]
colors_pie  = ['#E74C3C', '#2ECC71', '#3498DB', '#9B59B6']
wedges, texts, autotexts = ax2.pie(
    test_counts, labels=CLASS_NAMES, autopct='%1.1f%%',
    colors=colors_pie, startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts: t.set_fontsize(11)
ax2.set_title(f'Test Set Composition\n(Total: {sum(test_counts)} images)', fontsize=11)

plt.tight_layout()
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(f'{RESULTS_DIR}/overall_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSummary:")
for k, v in summary_metrics.items():
    bar = '█' * int(v * 20)
    print(f"  {k:<22}: {v:.4f}  {bar}")
print(f"\nDashboard saved to: {RESULTS_DIR} ✅")

## 🖼️ Step 18 — Sample Test Predictions

In [ ]:
model.eval()
inv_norm = transforms.Normalize(
    mean=[-m/s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1/s for s in IMAGENET_STD]
)

sample_loader = DataLoader(test_dataset, batch_size=20, shuffle=True)
images, labels = next(iter(sample_loader))

with torch.no_grad():
    outputs = model(images.to(DEVICE))
    preds   = outputs.argmax(dim=1).cpu()
    probs   = torch.softmax(outputs, dim=1).cpu()

fig, axes = plt.subplots(4, 5, figsize=(16, 13))
for idx, ax in enumerate(axes.flat):
    img = inv_norm(images[idx]).permute(1, 2, 0).clamp(0, 1).numpy()
    correct = (labels[idx] == preds[idx]).item()
    ax.imshow(img)
    ax.set_title(f"T:{CLASS_NAMES[labels[idx]]}\nP:{CLASS_NAMES[preds[idx]]}", color='green' if correct else 'red')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/sample_images.png', dpi=150)
plt.savefig(f'{RESULTS_DIR}/sample_predictions.png', dpi=150)
plt.show()

## 🧪 Step 19 — Out-of-Distribution (OOD) Evaluation
This section tests the model's robustness on an external dataset.

In [ ]:
from google.colab import files
import os
import subprocess

# 1. Manual Upload OOD Dataset (.rar)
print("Upload your OOD_Dataset.rar file:")
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded. Please run the cell again and select a file.")
else:
    ood_rar = list(uploaded.keys())[0]
    OOD_DATA_ROOT = '/content/data/OOD'

    # 2. Extract RAR
    # Install unrar if not present
    print("Installing unrar...")
    subprocess.run(['apt-get', 'install', '-y', 'unrar'], capture_output=True)

    os.makedirs(OOD_DATA_ROOT, exist_ok=True)
    print(f"\nExtracting '{ood_rar}' to {OOD_DATA_ROOT} using unrar...")

    # Use double quotes to handle filenames with spaces
    !unrar x -y "{ood_rar}" "{OOD_DATA_ROOT}/"

    print("\nOOD Extraction complete.")

In [ ]:
import gdown
import os

# 3. Download Main Dataset
main_file_id = '10KtvuwBoVo9Rj5jFurZ5-au0Yiive6-9'
main_url = f'https://drive.google.com/uc?id={main_file_id}'
main_rar = '/content/Organized_New.rar'

if not os.path.exists(main_rar):
    print("Downloading main dataset...")
    gdown.download(main_url, main_rar, quiet=False, fuzzy=True)

# 4. Extract Main Dataset
!apt-get install -y unrar
os.makedirs('/content/data', exist_ok=True)
print("\nExtracting main dataset...")
!unrar x -y {main_rar} /content/data/

# Update the root path for Step 6
DATASET_ROOT = '/content/data/Organized'
print(f"\nMain dataset extracted to: {DATASET_ROOT}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
import json

ACTUAL_OOD_DIR = None
for root, dirs, files in os.walk(OOD_DATA_ROOT):
    if any(cls in dirs for cls in CLASS_NAMES):
        ACTUAL_OOD_DIR = root
        break

if ACTUAL_OOD_DIR:
    # Updated to use train_transforms to apply your augmentation methodology to OOD data
    ood_dataset = datasets.ImageFolder(ACTUAL_OOD_DIR, transform=train_transforms)
    ood_loader = DataLoader(ood_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
    model.eval()

    ood_preds, ood_labels = [], []
    with torch.no_grad():
        for inputs, labels in ood_loader:
            outputs = model(inputs.to(DEVICE))
            ood_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            ood_labels.extend(labels.numpy())

    # Calculate Metrics
    ood_acc = accuracy_score(ood_labels, ood_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(ood_labels, ood_preds, average='weighted')
    report = classification_report(ood_labels, ood_preds, target_names=CLASS_NAMES, output_dict=True)

    # Save OOD results to the run-specific directory including total aggregate metrics
    ood_results = {
        "total_accuracy": float(ood_acc),
        "total_precision": float(precision),
        "total_recall": float(recall),
        "total_f1_score": float(f1),
        "per_class_metrics": {cls: report[cls] for cls in CLASS_NAMES}
    }

    with open(f'{RESULTS_DIR}/ood_performance_report.json', 'w') as f:
        json.dump(ood_results, f, indent=4)

    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(ood_labels, ood_preds), annot=True, fmt='d', cmap='Purples', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title("OOD Confusion Matrix (with Augmentations)")
    plt.savefig(f'{RESULTS_DIR}/ood_confusion_matrix.png', dpi=150)
    plt.show()

    print(f"OOD Accuracy: {ood_acc*100:.2f}%")
    print(f"OOD Weighted F1-Score: {f1:.4f}")
    print(f"OOD results saved to: {RESULTS_DIR} ✅")

## 💾 Step 20 — Save Final Model & All Artefacts

In [ ]:
FINAL_PATH = f'{CHECKPOINT_DIR}/final_resnet50_braintumor.pth'

# Save comprehensive final state
torch.save({
    'model_state': model.state_dict(),
    'history': history,
    'run_name': RUN_NAME,
    'test_accuracy': test_acc,
    'class_names': CLASS_NAMES
}, FINAL_PATH)

print('═'*60)
print(f' RUN: {RUN_NAME} - FINAL VERIFICATION')
print('═'*60)

# Detailed checklist of all expected files in the new Drive folder
expected_files = [
    ('Best Model', f'{CHECKPOINT_DIR}/best_resnet50_braintumor.pth'),
    ('Final Model', FINAL_PATH),
    ('Training History', f'{RESULTS_DIR}/history.json'),
    ('Training Curves', f'{RESULTS_DIR}/training_curves.png'),
    ('Class Distribution', f'{RESULTS_DIR}/class_distribution.png'),
    ('Layer distribution', f'{RESULTS_DIR}/layer_distribution.png'),
    ('Sample Images', f'{RESULTS_DIR}/sample_images.png'),
    ('Sample Predictions', f'{RESULTS_DIR}/sample_predictions.png'),
    ('Confusion Matrix', f'{RESULTS_DIR}/confusion_matrix.png'),
    ('Per-Class Metrics Bar', f'{RESULTS_DIR}/per_class_metrics.png'),
    ('Radar Chart', f'{RESULTS_DIR}/radar_chart.png'),
    ('Performance Dashboard', f'{RESULTS_DIR}/overall_dashboard.png'),
    ('Classification Report (TXT)', f'{RESULTS_DIR}/classification_report.txt'),
    ('Classification Report (JSON)', f'{RESULTS_DIR}/classification_report.json'),
    ('OOD Report', f'{RESULTS_DIR}/ood_performance_report.json'),
    ('OOD Confusion Matrix', f'{RESULTS_DIR}/ood_confusion_matrix.png'),
]

for label, path in expected_files:
    status = '✅' if os.path.exists(path) else '❌ (missing)'
    print(f"  {status}  {label:<30}")

print('═'*60)
print(f"Location: /content/drive/MyDrive/BrainTumor/{RUN_NAME}/")